In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/raw/results.csv")

In [4]:
df["date"] = pd.to_datetime(df["date"])

In [5]:
completed_matches = df[
    df["home_team"].notna() &
    df["away_team"].notna()
].copy()

completed_matches = completed_matches.sort_values("date")

In [6]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    elif row["home_score"] < row["away_score"]:
        return "A"
    return "D"

completed_matches["result"] = completed_matches.apply(get_result, axis=1)

In [7]:
completed_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,D
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,H
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,H
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,D
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,H


In [8]:
def get_team_matches(team, date, matches):

    previous_matches = matches[
        (
            (matches["home_team"] == team) |
            (matches["away_team"] == team)
        )
        &
        (matches["date"] < date)
    ]

    return previous_matches.sort_values("date", ascending = False)

In [9]:
get_team_matches("Argentina", pd.Timestamp("2026-01-01"), completed_matches).head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
48893,2025-11-14,Angola,Argentina,0.0,2.0,Friendly,Luanda,Angola,False,A
48816,2025-10-14,Puerto Rico,Argentina,0.0,6.0,Friendly,Fort Lauderdale,United States,True,A
48730,2025-10-10,Argentina,Venezuela,1.0,0.0,Friendly,Miami Gardens,United States,True,H
48646,2025-09-09,Ecuador,Argentina,1.0,0.0,FIFA World Cup qualification,Guayaquil,Ecuador,False,H
48533,2025-09-04,Argentina,Venezuela,3.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H


In [10]:
def get_recent_matches(team, date, matches, n=10):
    previous_matches = get_team_matches(team, date, matches)
    return previous_matches.head(n)

In [16]:
argentina_recent = get_recent_matches("Argentina", pd.Timestamp("2026-01-01"), completed_matches)

argentina_recent.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
48893,2025-11-14,Angola,Argentina,0.0,2.0,Friendly,Luanda,Angola,False,A
48816,2025-10-14,Puerto Rico,Argentina,0.0,6.0,Friendly,Fort Lauderdale,United States,True,A
48730,2025-10-10,Argentina,Venezuela,1.0,0.0,Friendly,Miami Gardens,United States,True,H
48646,2025-09-09,Ecuador,Argentina,1.0,0.0,FIFA World Cup qualification,Guayaquil,Ecuador,False,H
48533,2025-09-04,Argentina,Venezuela,3.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H
48438,2025-06-10,Argentina,Colombia,1.0,1.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,D
48329,2025-06-05,Chile,Argentina,0.0,1.0,FIFA World Cup qualification,Santiago,Chile,False,A
48260,2025-03-25,Argentina,Brazil,4.0,1.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H
48170,2025-03-21,Uruguay,Argentina,0.0,1.0,FIFA World Cup qualification,Montevideo,Uruguay,False,A
48017,2024-11-19,Argentina,Peru,1.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H


In [17]:
def calculate_win_rate(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if len(recent_matches) == 0:
        return np.nan
    
    wins = 0

    for _, match in recent_matches.iterrows():

        if(
            match["home_team"] == team 
            and match["home_score"] > match["away_score"]
        ):
            wins += 1
        
        elif(
            match["away_team"] == team 
            and match["away_score"] > match["home_score"]
        ):
            wins += 1
    
    return wins/len(recent_matches)

In [21]:
print("Argentina:", calculate_win_rate("Argentina", pd.Timestamp("2026-01-01"), completed_matches))

print("Brazil:", calculate_win_rate("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 0.8
Brazil: 0.5


In [22]:
def calculate_avg_goals_scored(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if(len(recent_matches) == 0):
        return np.nan

    goals = []

    for _, match in recent_matches.iterrows():
        if match["home_team"] == team:
            goals.append(match["home_score"])
        
        elif match["away_team"] == team:
            goals.append(match["away_score"])
    
    return np.mean(goals)

In [28]:
print("Argentina:", calculate_avg_goals_scored("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_avg_goals_scored("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 2.0
Brazil: 1.7


In [24]:
def calculate_avg_goals_conceded(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if(len(recent_matches) == 0):
        return np.nan

    goals = []

    for _, match in recent_matches.iterrows():
        if match["home_team"] == team:
            goals.append(match["away_score"])
        
        elif match["away_team"] == team:
            goals.append(match["home_score"])
    
    return np.mean(goals)

In [29]:
print("Argentina:", calculate_avg_goals_conceded("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_avg_goals_conceded("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 0.3
Brazil: 1.0


In [26]:
def calculate_goal_difference(team, date, matches, n = 10):
    scored_goals = calculate_avg_goals_scored(team, date, matches, n)
    conceded_goals = calculate_avg_goals_conceded(team, date, matches, n)

    return scored_goals - conceded_goals

In [30]:
print("Argentina:", calculate_goal_difference("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_goal_difference("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 1.7
Brazil: 0.7


In [ ]:
print("Argentina:", calculate_avg_goals_scored("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_avg_goals_scored("Brazil", pd.Timestamp("2026-01-01"), completed_matches))